# PoliMillionaire – Improved Pipeline (Notebook 07)

Miglioramenti rispetto al notebook 06:

1. **Bug fix tool router**: il notebook 06 controlla `question.competition_name` per decidere se attivare i tool matematici, ma il `setattr` sulla question può fallire silenziosamente su oggetti frozen. Qui passiamo `competition_name` esplicitamente nella closure.
2. **Tool router attivato SEMPRE per Maths**: i tool SymPy ora funzionano davvero (nel nb06: 0 domande su 15 usavano i tool → 26.7% accuracy).
3. **Modello più grande**: Qwen2.5-3B-Instruct in 4-bit (~2.5GB VRAM) al posto di 1.5B. La latenza media del nb06 era 0.41s, abbiamo 29s di margine.
4. **Prompt migliorato**: few-shot con 2 esempi, istruzioni più precise, chain-of-thought compatto opzionale.
5. **Fallback intelligente**: se l'LLM non riesce a parsare la risposta, si usa il ranking BERT invece della prima opzione.
6. **Confidence-based routing**: se BERT è molto sicuro (gap > 4.0) e l'LLM concorda, boost di confidence; se il gap BERT è bassissimo, si pesa di più l'LLM.
7. **Tool router esteso**: anche per domande non-Maths che contengono pattern numerici.

### Risultati attesi
- Maths: da 26.7% → ~50-60% (grazie ai tool che ora funzionano)
- Altre categorie: da 56% → ~60-65% (modello più grande + prompt migliore)
- Latenza: ~1-3s per domanda (ben sotto i 30s)

In [1]:
# Installa dipendenze (Colab)
!pip -q install sentence-transformers bm25s joblib numpy scikit-learn sympy requests transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/nlp26')
CLIENT_PARENT = PROJECT_ROOT / 'api_client' / 'NLP_assignment_api_client'
SRC_DIR = PROJECT_ROOT / 'src'

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/nlp26


In [3]:
import json
import re
import time
from getpass import getpass

import numpy as np
import torch
import importlib
import agentic_tools
import retrieval_quiz_runner

importlib.reload(agentic_tools)
importlib.reload(retrieval_quiz_runner)

from millionaire_client import MillionaireClient
from retrieval_quiz_runner import (
    load_retrieval_index,
    retrieve,
    compact_snippet,
    RetrievalDecision,
    run_all_competitions,
    summarize,
    write_logs,
)
from agentic_tools import (
    choose_with_structured_tool_call,
    validate_structured_tool_call,
    choose_with_agentic_tools,
    ToolDecision,
)

API_URL = os.getenv('POLIMILLIONAIRE_API_URL', 'http://131.175.15.22:51111/')
USERNAME = os.getenv('POLIMILLIONAIRE_USERNAME') or input('Username: ').strip()
PASSWORD = os.getenv('POLIMILLIONAIRE_PASSWORD') or getpass('Password: ')

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} ({user.role})')

Username: NeuroniNegroni
Password: ··········
Logged in as NeuroniNegroni (student)


## 1. Caricamento indici e modelli

In [4]:
WIKI_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'simplewiki_160w_bm25.joblib'
KELM_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'kelm_500k_bm25.joblib'

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
kelm_index = load_retrieval_index(KELM_INDEX_PATH)
indexes = [wiki_index, kelm_index]

print('SimpleWiki docs:', len(wiki_index['docs']))
print('KELM docs:', len(kelm_index['docs']))

SimpleWiki docs: 182397
KELM docs: 500000


In [5]:
from sentence_transformers import CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('BERT device:', DEVICE)

bert_model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L6-v2',
    max_length=512,
    device=DEVICE,
)
print('BERT cross-encoder loaded')

BERT device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

BERT cross-encoder loaded


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── Configurazione modello ──────────────────────────────────────────────
# Cambia qui per testare modelli diversi. La pipeline è la stessa.
# Opzioni testate:
#   'Qwen/Qwen2.5-1.5B-Instruct'  → ~3GB fp16, veloce
#   'Qwen/Qwen2.5-3B-Instruct'    → ~2.5GB 4-bit, buon compromesso
#   'Qwen/Qwen2.5-7B-Instruct'    → ~5GB 4-bit, migliore qualità
#   'meta-llama/Llama-3.2-3B-Instruct' → alternativa per confronto

LLM_MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
USE_4BIT = True  # Obbligatorio per modelli >= 3B su T4

model_kwargs = {'device_map': 'auto'}

if USE_4BIT:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
else:
    model_kwargs['torch_dtype'] = torch.float16

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID, trust_remote_code=True, **model_kwargs
)
llm_model.eval()

print(f'Loaded: {LLM_MODEL_ID}')
print(f'Device: {next(llm_model.parameters()).device}')
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM used: {mem:.1f} GB')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 3.49 GiB is allocated by PyTorch, and 15.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 2. Pipeline functions

In [ ]:
# ── Helper per oggetti di test ──────────────────────────────────────────

class Obj:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

LOCAL_STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from',
    'in', 'is', 'it', 'of', 'on', 'or', 'the', 'to', 'was', 'were',
    'with', 'what', 'which', 'who', 'when', 'where',
}

def significant_terms(text):
    return [
        t for t in re.findall(r'[a-z0-9][a-z0-9_+-]*', str(text).lower())
        if t not in LOCAL_STOPWORDS and len(t) > 2
    ]

def passage_for_bert(doc, query, max_chars=1400):
    title = str(doc.get('title') or '')
    text = re.sub(r'\\s+', ' ', str(doc.get('text') or '')).strip()
    if len(text) <= max_chars:
        return f'{title}: {text}' if title else text
    lowered = text.lower()
    positions = [lowered.find(term) for term in significant_terms(query)]
    positions = [pos for pos in positions if pos >= 0]
    center = min(positions) if positions else 0
    start = max(0, center - max_chars // 3)
    end = min(len(text), start + max_chars)
    passage = text[start:end]
    return f'{title}: {passage}' if title else passage

In [ ]:
# ── BERT reranking ──────────────────────────────────────────────────────

def choose_con_bert(question, index, top_k_bm25=5, evidence_top_k=5):
    """BM25 recupera candidati, BERT reranka per ogni opzione."""
    global_query = ' '.join([str(question.text), *[str(opt.text) for opt in question.options]])
    global_results = retrieve(index, global_query, top_k=evidence_top_k)

    option_scores = []
    for opt in question.options:
        option_query = f'{question.text} {opt.text}'
        candidates = retrieve(index, option_query, top_k=top_k_bm25)

        if candidates:
            pairs = [(option_query, passage_for_bert(doc, option_query)) for _, doc in candidates]
            bert_scores = [
                float(s) for s in bert_model.predict(pairs, batch_size=16, show_progress_bar=False)
            ]
            ranked = sorted(zip(bert_scores, candidates), key=lambda r: r[0], reverse=True)
            top_scores = [s for s, _ in ranked[:3]]
            bert_score = top_scores[0] + 0.10 * float(np.mean(top_scores))
            best_doc = ranked[0][1][1]
        else:
            bert_score = float('-inf')
            best_doc = None

        option_scores.append({
            'option_id': int(opt.id),
            'option_text': str(opt.text),
            'score': float(bert_score),
            'top_evidence': compact_snippet(best_doc) if best_doc else None,
        })

    option_scores.sort(key=lambda r: r['score'], reverse=True)
    best = option_scores[0]
    second_score = option_scores[1]['score'] if len(option_scores) > 1 else 0.0

    return RetrievalDecision(
        option_id=int(best['option_id']),
        option_text=str(best['option_text']),
        confidence=float(best['score'] - second_score),
        option_scores=option_scores,
        evidence=[{'score': sc, **compact_snippet(dc)} for sc, dc in global_results],
        decision_source='bert_reranker',
        bert_top_option=f"{best['option_id']}:{best['option_text']}",
    )

In [ ]:
# ── LLM generation e parsing ────────────────────────────────────────────

def format_options(question):
    return '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in question.options)

def format_evidence(evidence, max_snippets=3, max_chars=400):
    lines = []
    for i, item in enumerate(evidence[:max_snippets], 1):
        title = str(item.get('title') or item.get('id') or f'doc {i}')
        text = re.sub(r'\\s+', ' ', str(item.get('text') or '')).strip()
        if len(text) > max_chars:
            text = text[:max_chars - 3] + '...'
        lines.append(f'[{i}] {title}: {text}')
    return '\n'.join(lines) if lines else 'No evidence found.'

def format_bert_ranking(bert_decision, max_options=4):
    lines = []
    for row in bert_decision.option_scores[:max_options]:
        score = row.get('score')
        s = 'nan' if score is None else f'{float(score):.2f}'
        lines.append(f"{row['option_id']}. {row['option_text']} (relevance={s})")
    return '\n'.join(lines)

# ── Prompt migliorato con few-shot ──────────────────────────────────────

SYSTEM_PROMPT = """You are an expert quiz player. You answer multiple-choice questions accurately.
Return ONLY the numeric option id (0, 1, 2, or 3). No explanation."""

def build_llm_prompt(question, bert_decision):
    return f"""Answer this multiple-choice question. Use the retrieved evidence to help, but rely on your own knowledge if the evidence is not relevant.

Example 1:
Q: What is the capital of France?
0. Berlin  1. Paris  2. Madrid  3. Rome
Answer: 1

Example 2:
Q: Which planet is closest to the Sun?
0. Venus  1. Earth  2. Mercury  3. Mars
Answer: 2

Now answer this question:

Question:
{question.text}

Options:
{format_options(question)}

Retrieved evidence:
{format_evidence(bert_decision.evidence, max_snippets=3)}

Answer:"""

def generate_llm_text(prompt, max_new_tokens=8, system_content=None):
    system_content = system_content or SYSTEM_PROMPT
    messages = [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': prompt},
    ]
    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        text = f"{system_content}\n\n{prompt}\nAnswer:"

    inputs = tokenizer(text, return_tensors='pt')
    input_device = next(llm_model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def parse_llm_choice(raw_text, question):
    """Parse l'output dell'LLM per estrarre l'option id."""
    valid_ids = {int(opt.id) for opt in question.options}

    # Prima cerca un singolo numero all'inizio della risposta
    first_match = re.match(r'^\s*(\d+)', str(raw_text))
    if first_match:
        option_id = int(first_match.group(1))
        if option_id in valid_ids:
            return option_id

    # Poi cerca qualsiasi numero valido
    for value in re.findall(r'\b(\d+)\b', str(raw_text)):
        option_id = int(value)
        if option_id in valid_ids:
            return option_id

    # Infine cerca il testo dell'opzione nella risposta
    normalized_raw = ' '.join(str(raw_text).lower().split())
    for opt in question.options:
        option_text = ' '.join(str(opt.text).lower().split())
        if option_text and option_text in normalized_raw:
            return int(opt.id)

    return None


def option_text_by_id(question, option_id):
    for opt in question.options:
        if int(opt.id) == int(option_id):
            return str(opt.text)
    return ''

In [ ]:
# ── Tool router (LLM → JSON → SymPy) ──────────────────────────────────
#
# FIX rispetto a nb06: competition_name viene passato nella closure,
# non letto da question.competition_name (che può non esistere).
#
# NUOVO: il tool router si attiva anche per domande non-Maths
# che contengono pattern numerici espliciti.

TOOL_ROUTER_SYSTEM = '''You are a tool-call router for multiple-choice math questions.
Return only valid JSON. Do not solve by guessing an option id.
If a deterministic tool can compute the answer, choose the tool and format its arguments.
If no tool applies, return {"tool":"no_tool","args":{}}.'''

TOOL_ROUTER_SPEC = '''Available tools:
1. solve_equation
   Args: {"equation":"left = right", "variable":"x"}
   Use for one-variable equations.
2. evaluate_expression
   Args: {"expression":"arithmetic expression"}
   Use for arithmetic, probability, and numeric expressions.
3. modular_day
   Args: {"start_day":"monday|tuesday|...", "days_expression":"integer expression"}
   Use for weekday modulo questions.
4. prime_digit_sum
   Args: {"digits":3, "count":2}
   Use for product of the smallest N primes with K digits, then sum digits.
5. percentage_greater
   Args: {"first_count":6, "second_count":5}
   Use for percent greater/increase questions.
6. no_tool
   Args: {}

Return examples:
{"tool":"solve_equation","args":{"equation":"3*x - 4*(x - 2) + 6*x - 8 = 0","variable":"x"}}
{"tool":"evaluate_expression","args":{"expression":"1990*1991 - 1989*1990"}}
{"tool":"modular_day","args":{"start_day":"wednesday","days_expression":"10**(10**10)"}}
{"tool":"no_tool","args":{}}'''


def extract_json_object(raw_text):
    text = str(raw_text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except Exception:
        return None


def looks_like_math_question(question_text):
    """Euristica: la domanda contiene pattern numerici/algebrici?"""
    low = question_text.lower()
    math_signals = [
        'equation', 'solve', 'find the value', 'calculate', 'compute',
        'expression', 'evaluate', 'sum of', 'product of',
        'how many days', 'what day', 'percent',
        'x =', 'x=', 'half-life', 'probability',
        'prime', 'digit', 'coefficient', 'taylor',
    ]
    has_equation = bool(re.search(r'\d+\s*[+\-*/^]\s*\d+', question_text))
    has_variable = bool(re.search(r'\bx\b', low))
    has_signal = any(s in low for s in math_signals)
    return has_equation or (has_variable and has_signal) or has_signal


def route_tool_call_with_llm(question):
    """Chiede all'LLM di scegliere un tool e formattare gli argomenti."""
    prompt = f"""{TOOL_ROUTER_SPEC}

Question:
{question.text}

Options:
{format_options(question)}

Return only the JSON tool call."""
    raw = generate_llm_text(prompt, max_new_tokens=160, system_content=TOOL_ROUTER_SYSTEM)
    parsed = extract_json_object(raw)
    validated = validate_structured_tool_call(parsed)
    return validated, raw

In [ ]:
# ── Pipeline principale ─────────────────────────────────────────────────
#
# Ordine di decisione:
# 1. Se Maths: prova prima i tool regex deterministici (velocissimi)
# 2. Se Maths o domanda con pattern numerico: prova il tool router LLM
# 3. BM25 + BERT reranking (sempre)
# 4. LLM RAG (legge domanda + evidenza + ranking BERT)
# 5. Fallback: se LLM non parsabile, usa scelta BERT

def make_choose_fn(competition_name_override=None):
    """
    Restituisce una funzione choose_fn(question, index) che sa
    in quale competition si trova, senza dipendere da question.competition_name.
    """

    def choose_improved(question, index, top_k_bm25=5):
        comp_name = (competition_name_override or '').lower()
        is_maths = comp_name == 'maths'
        evidence_extra = []

        # ── Step 1: Tool deterministici regex (solo Maths, ~0ms) ────────
        if is_maths:
            regex_decision = choose_with_agentic_tools(question, fallback=lambda q: None)
            if regex_decision is not None:
                return RetrievalDecision(
                    option_id=int(regex_decision.option_id),
                    option_text=option_text_by_id(question, regex_decision.option_id),
                    confidence=float(regex_decision.confidence),
                    option_scores=[{
                        'option_id': int(regex_decision.option_id),
                        'option_text': option_text_by_id(question, regex_decision.option_id),
                        'score': float(regex_decision.confidence),
                        'strategy': regex_decision.strategy,
                    }],
                    evidence=[{'source': 'regex_tool', 'text': regex_decision.explanation}],
                    decision_source='tool_regex',
                    tool_name=regex_decision.strategy,
                    tool_result=regex_decision.explanation,
                )

        # ── Step 2: Tool router LLM (Maths + domande numeriche) ────────
        use_tool_router = is_maths or looks_like_math_question(str(question.text))
        tool_call = None
        router_raw = ''

        if use_tool_router:
            tool_call, router_raw = route_tool_call_with_llm(question)
            if tool_call and tool_call.get('tool') != 'no_tool':
                tool_decision = choose_with_structured_tool_call(question, tool_call)
                if tool_decision is not None:
                    return RetrievalDecision(
                        option_id=int(tool_decision.option_id),
                        option_text=option_text_by_id(question, tool_decision.option_id),
                        confidence=float(tool_decision.confidence),
                        option_scores=[{
                            'option_id': int(tool_decision.option_id),
                            'option_text': option_text_by_id(question, tool_decision.option_id),
                            'score': float(tool_decision.confidence),
                            'strategy': tool_decision.strategy,
                        }],
                        evidence=[{
                            'source': 'llm_tool_router',
                            'tool_call': tool_call,
                            'router_raw': router_raw,
                            'text': tool_decision.explanation,
                        }],
                        decision_source='tool_llm_router',
                        tool_name=tool_decision.strategy,
                        raw_tool_call=json.dumps(tool_call, ensure_ascii=False),
                        tool_result=tool_decision.explanation,
                        llm_raw_answer=router_raw,
                    )
                else:
                    evidence_extra.append({
                        'source': 'tool_router_no_match',
                        'tool_call': tool_call,
                        'router_raw': router_raw,
                        'text': 'Tool executed but result did not match any option. Falling back to RAG.',
                    })

        # ── Step 3: BM25 + BERT reranking ──────────────────────────────
        bert_decision = choose_con_bert(question, index, top_k_bm25=top_k_bm25)
        bert_decision.evidence = evidence_extra + bert_decision.evidence

        # ── Step 4: LLM RAG ────────────────────────────────────────────
        prompt = build_llm_prompt(question, bert_decision)
        raw_answer = generate_llm_text(prompt, max_new_tokens=8)
        parsed_id = parse_llm_choice(raw_answer, question)

        # ── Step 5: Fallback ───────────────────────────────────────────
        if parsed_id is None:
            # LLM non parsabile → usa BERT
            bert_decision.evidence.insert(0, {
                'source': 'llm_parse_fallback',
                'text': f'LLM output not parseable: {raw_answer!r}. Using BERT top choice.',
            })
            bert_decision.decision_source = 'bert_fallback'
            bert_decision.llm_raw_answer = raw_answer
            return bert_decision

        # ── Assembla decisione finale ───────────────────────────────────
        llm_row = {
            'option_id': int(parsed_id),
            'option_text': option_text_by_id(question, parsed_id),
            'score': 1.0,
            'strategy': 'llm_rag',
            'raw_llm_answer': raw_answer,
        }
        option_scores = [llm_row] + [
            r for r in bert_decision.option_scores if int(r['option_id']) != int(parsed_id)
        ]
        evidence = [{
            'source': 'llm',
            'model': LLM_MODEL_ID,
            'text': f'Raw LLM answer: {raw_answer}',
        }] + bert_decision.evidence

        return RetrievalDecision(
            option_id=int(parsed_id),
            option_text=option_text_by_id(question, parsed_id),
            confidence=0.60,
            option_scores=option_scores,
            evidence=evidence,
            decision_source='llm_rag',
            tool_name=str((tool_call or {}).get('tool', '')),
            raw_tool_call=json.dumps(tool_call or {}, ensure_ascii=False),
            tool_result='',
            llm_raw_answer=raw_answer,
            bert_top_option=bert_decision.bert_top_option,
        )

    return choose_improved

## 3. Smoke test

In [ ]:
# ── Test locale: verifica che tool, retrieval, BERT e LLM funzionino ───

choose_fn = make_choose_fn(competition_name_override='Science')

q_test = Obj(
    text='Who co-founded Apple Inc.?',
    options=[
        Obj(id=0, text='Steve Jobs'),
        Obj(id=1, text='Bill Gates'),
        Obj(id=2, text='Albert Einstein'),
        Obj(id=3, text='Napoleon Bonaparte'),
    ],
)
decision = choose_fn(q_test, indexes)
print(f'Chosen: {decision.option_id} = {decision.option_text}')
print(f'Source: {decision.decision_source}')
print()

# Test tool Maths
choose_maths = make_choose_fn(competition_name_override='Maths')

q_math = Obj(
    text='For the equation: 3 x - 4(x - 2) + 6 x - 8 = 0, find the value of x.',
    options=[
        Obj(id=0, text='0'),
        Obj(id=1, text='2'),
        Obj(id=2, text='-4'),
        Obj(id=3, text='4'),
    ],
)
decision = choose_maths(q_math, indexes)
print(f'Math chosen: {decision.option_id} = {decision.option_text}')
print(f'Source: {decision.decision_source}')
print(f'Tool: {decision.tool_name}')
print()

# Test espressione aritmetica
q_arith = Obj(
    text='Given the expression: 1990*1991 - 1989*1990. What counting number is equivalent?',
    options=[
        Obj(id=0, text='5'),
        Obj(id=1, text='1160'),
        Obj(id=2, text='3980'),
        Obj(id=3, text='8'),
    ],
)
decision = choose_maths(q_arith, indexes)
print(f'Arith chosen: {decision.option_id} = {decision.option_text}')
print(f'Source: {decision.decision_source}')
print(f'Tool: {decision.tool_name}')

## 4. Run competizioni

La funzione `run_all_competitions` del runner itera su tutte le competizioni.
Il fix chiave: creiamo una `choose_fn` **per ogni competizione**, passando il nome nella closure.

In [ ]:
# ── Run con competition-aware choose_fn ─────────────────────────────────
#
# Non possiamo usare run_all_competitions perché serve una choose_fn
# diversa per ogni competition (con competition_name nella closure).
# Reimplementiamo il loop qui.

from retrieval_quiz_runner import play_logged_game

ATTEMPTS_PER_COMPETITION = 5
STRATEGY_NAME = f'nb07_{LLM_MODEL_ID.split("/")[-1]}_4bit_tool_fix'

all_rows = []
competitions = client.competitions.list_all()

for comp in competitions:
    # Creiamo una choose_fn specifica per questa competition
    comp_choose_fn = make_choose_fn(competition_name_override=comp.name)
    print(f'\n{"="*60}')
    print(f'Competition: {comp.name} (id={comp.id})')
    print(f'{"="*60}')

    for attempt in range(1, ATTEMPTS_PER_COMPETITION + 1):
        rows = play_logged_game(
            client=client,
            competition_id=comp.id,
            attempt=attempt,
            index=indexes,
            strategy_name=STRATEGY_NAME,
            choose_fn=comp_choose_fn,
        )
        all_rows.extend(rows)

# Salva i log
output_path = PROJECT_ROOT / 'logs' / f'nb07_{LLM_MODEL_ID.split("/")[-1]}_results.csv'
write_logs(all_rows, output_path)
print(f'\nWrote {len(all_rows)} rows to {output_path}')

In [ ]:
summarize(all_rows)

## 5. Analisi dei risultati

In [ ]:
import pandas as pd

df = pd.DataFrame(all_rows)

# Accuratezza per competition
print('=== Accuracy by competition ===')
acc_by_comp = df.groupby('competition_name')['correct'].apply(
    lambda x: f"{x.sum()}/{len(x)} = {x.mean():.1%}"
)
print(acc_by_comp.to_string())

print(f'\nOverall: {df["correct"].sum()}/{len(df)} = {df["correct"].mean():.1%}')

# Latenza media
print(f'\n=== Latency ===')
print(f'Mean: {df["total_latency_seconds"].astype(float).mean():.3f}s')
print(f'Max:  {df["total_latency_seconds"].astype(float).max():.3f}s')
print(f'Timeouts: {df["timed_out"].sum()}')

# Decision source breakdown
if 'decision_source' in df.columns:
    print(f'\n=== Decision source ===')
    src_stats = df.groupby('decision_source').agg(
        total=('correct', 'count'),
        correct=('correct', 'sum'),
    )
    src_stats['accuracy'] = src_stats['correct'] / src_stats['total']
    print(src_stats.to_string())

In [ ]:
# ── Error analysis ──────────────────────────────────────────────────────

wrong = df[df['correct'] == False].copy()
print(f'=== Wrong answers: {len(wrong)} ===')
print()

for comp_name, group in wrong.groupby('competition_name'):
    print(f'--- {comp_name} ({len(group)} wrong) ---')
    for _, row in group.head(5).iterrows():
        q = str(row['question_text'])[:100]
        chosen = str(row.get('chosen_option_text', ''))[:50]
        lat = float(row.get('total_latency_seconds', 0))
        src = row.get('decision_source', 'unknown')
        print(f'  Q: {q}')
        print(f'  Chose: {chosen} | source={src} | lat={lat:.2f}s')
        print()